In [11]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import optuna

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    roc_curve, confusion_matrix, recall_score
)

import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from aml.data.load import load_data

In [12]:
df = load_data()

## Feature Engineering

In [13]:
df["is_currency_same"] = (df["Receiving Currency"] == df["Payment Currency"]).astype("int8")
df["is_same_bank"] = (df["From Bank"] == df["To Bank"]).astype("int8")
df["is_self_transfer"] = (df["Account"] == df["Account.1"]).astype("int8")

# time
df["hour_sin"] = np.sin(2 * np.pi * df.hour / 24).astype("float32") 
df["hour_cos"] = np.cos(2 * np.pi * df.hour / 24).astype("float32")
df["dow_sin"] = np.sin(2 * np.pi * df.day / 7).astype("float32")
df["dow_cos"] = np.cos(2 * np.pi * df.day / 7).astype("float32")
df["is_night"] = df.hour.between(0, 5.999).astype("int8")
df["is_weekend"] = df.day.isin([5, 6]).astype("int8")

# amaunt
paid = df["Amount Paid"].clip(lower=0)
received = df["Amount Received"].clip(lower=0)
df["log_amount_paid"] = np.log1p(paid).astype("float32")
df["log_amount_received"] = np.log1p(received).astype("float32")
df["amount_log_gap"] = np.abs(df["log_amount_paid"] - df["log_amount_received"]).astype("float32")
df["is_round_10"] = np.isclose(np.mod(paid, 10), 0, atol=1e-8).astype("int8")
df["is_round_100"] = np.isclose(np.mod(paid, 100), 0, atol=1e-8).astype("int8")
df["is_round_1000"] = np.isclose(np.mod(paid, 1000), 0, atol=1e-8).astype("int8")

In [14]:
sender = df["Account"]
receiver = df["Account.1"]

# velocity (количество прошлых транзакций)
sender_prev = df.groupby(sender, sort=False).cumcount().astype("int32")
receiver_prev = df.groupby(receiver, sort=False).cumcount().astype("int32")
pair_prev = df.groupby([sender, receiver], sort=False).cumcount().astype("int32")

df["sender_prev_tx_log"] = np.log1p(sender_prev).astype("float32")
df["receiver_prev_tx_log"] = np.log1p(receiver_prev).astype("float32")
df["pair_prev_tx_log"] = np.log1p(pair_prev).astype("float32")

new_pair = (pair_prev == 0).astype("int32") 
df["is_new_pair"] = new_pair.astype("int8")

# Fan-out / Fan-in (сетевое разрастание: уникальные контрагенты до текущего момента)
sender_unique_before = new_pair.groupby(sender, sort=False).cumsum() - new_pair
receiver_unique_before = new_pair.groupby(receiver, sort=False).cumsum() - new_pair

df["sender_unique_receivers_log"] = np.log1p(sender_unique_before).astype("float32")
df["receiver_unique_senders_log"] = np.log1p(receiver_unique_before).astype("float32")

# Доля межбаковских переводов
is_same_bank = (df["To Bank"] != df["From Bank"]).astype("int32")
is_same_bank.groupby(df["Account"]).cumcount()-is_same_bank
df["sender_prior_interbank_ratio"] = (
    (is_same_bank / sender_prev.replace(0, np.nan))
    .fillna(0)
    .astype("float32")
)

# Временная плотность (минуты с прошлой транзакции)
sender_dt = df.groupby(sender, sort=False)["Timestamp"].diff().dt.total_seconds() / 60
receiver_dt = df.groupby(receiver, sort=False)["Timestamp"].diff().dt.total_seconds() / 60

df["sender_minutes_since_prev_log"] = np.log1p(sender_dt.clip(lower=0)).fillna(-1).astype("float32")
df["receiver_minutes_since_prev_log"] = np.log1p(receiver_dt.clip(lower=0)).fillna(-1).astype("float32")

sender_dt_clean = sender_dt.fillna(0)
sender_dt_sum = sender_dt_clean.groupby(sender, sort=False).cumcount() - sender_dt_clean
sender_prior_avg_dt = sender_dt_sum / sender_prev.replace(0, np.nan)
df["sender_prior_avg_minutes_log"] = (
    np.log1p(sender_prior_avg_dt.clip(lower=0))
    .fillna(-1)
    .astype("float32")
)

# Behavioral Deviation (отклонение от исторического профиля)
sender_prior_sum = (paid.groupby(sender, sort=False).cumsum() - paid)
sender_prior_mean = sender_prior_sum / sender_prev.replace(0, np.nan)
amount_ratio = paid / sender_prior_mean.replace(0, np.nan)

df["amount_to_sender_prior_mean_log"] = (
    np.log1p(amount_ratio.clip(lower=0, upper=1e6))
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
      .astype("float32")
)

## Split

In [15]:
TRAIN_SIZE = 0.7
VALID_SHARE = 0.15

train_cut = df.Timestamp.quantile(TRAIN_SIZE)
val_cut = df.Timestamp.quantile(TRAIN_SIZE + VALID_SHARE)

train_mask = df.Timestamp < train_cut
val_mask = (df.Timestamp >= train_cut) & (df.Timestamp < val_cut)
test_mask = df.Timestamp > val_cut

df = df.drop(columns="Timestamp")

traint_df = df.iloc[np.flatnonzero(train_mask)]
val_df = df.iloc[np.flatnonzero(val_mask)]
test_df = df.iloc[np.flatnonzero(test_mask)]

In [16]:
neg_subsample_ratio = 0.3900

X_train, y_train = traint_df.drop(columns='Is Laundering'), traint_df['Is Laundering']
X_val, y_val = val_df.drop(columns='Is Laundering'), val_df['Is Laundering']
X_test, y_test = test_df.drop(columns='Is Laundering'), test_df['Is Laundering']

pos_mask = y_train == 1
neg_mask = y_train == 0

pos_index = y_train.index[pos_mask]
neg_index = y_train.index[neg_mask]

n_total_neg = len(neg_index)

if neg_subsample_ratio < 1.0:
    rng = np.random.default_rng(42)
    n_sample_neg = max(1, int(neg_subsample_ratio * n_total_neg))
    sampled_neg_idx = rng.choice(neg_index, size=n_sample_neg, replace=False)
    neg_weight = float(n_total_neg) / n_sample_neg
else:
    neg_weight = 1.0
    sampled_neg_idx = neg_index

fit_idx = np.concatenate([pos_index, sampled_neg_idx])

X_fit = X_train.loc[fit_idx]
y_fit = y_train.loc[fit_idx]

sample_weight = np.where(y_fit.to_numpy() == 0, neg_weight, 1.0).astype(np.float64)

## Model

Index(['From Bank', 'Account', 'To Bank', 'Account.1', 'Receiving Currency',
       'Payment Currency', 'Payment Format'],
      dtype='object')

In [25]:
cat = CatBoostClassifier()
cat.fit(X_fit, y_fit, 
        cat_features=[col for col in X_fit.select_dtypes(include=['string', 'object']).columns],
        eval_set=(X_val, y_val),
        early_stopping_rounds=200
        )

Learning rate set to 0.204306
0:	learn: 0.2280213	test: 0.2271736	best: 0.2271736 (0)	total: 367ms	remaining: 6m 6s
1:	learn: 0.0840889	test: 0.0823273	best: 0.0823273 (1)	total: 740ms	remaining: 6m 9s
2:	learn: 0.0374854	test: 0.0301372	best: 0.0301372 (2)	total: 1.05s	remaining: 5m 49s
3:	learn: 0.0167458	test: 0.0122052	best: 0.0122052 (3)	total: 1.78s	remaining: 7m 24s
4:	learn: 0.0102234	test: 0.0070636	best: 0.0070636 (4)	total: 2.41s	remaining: 8m
5:	learn: 0.0079962	test: 0.0051765	best: 0.0051765 (5)	total: 3.07s	remaining: 8m 29s
6:	learn: 0.0068843	test: 0.0043855	best: 0.0043855 (6)	total: 3.7s	remaining: 8m 45s
7:	learn: 0.0064152	test: 0.0039769	best: 0.0039769 (7)	total: 4.35s	remaining: 8m 59s
8:	learn: 0.0062219	test: 0.0037403	best: 0.0037403 (8)	total: 4.96s	remaining: 9m 6s
9:	learn: 0.0058884	test: 0.0034834	best: 0.0034834 (9)	total: 5.24s	remaining: 8m 39s
10:	learn: 0.0057706	test: 0.0033770	best: 0.0033770 (10)	total: 5.9s	remaining: 8m 50s
11:	learn: 0.0056795

CatBoostClassifier()

In [26]:
scores = cat.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, scores), average_precision_score(y_test, scores)

(0.9742186861502469, 0.3533007415400264)

In [8]:
account_cols = ['Account', 'Account.1', 'From Bank', 'To Bank']

numeric_features = [
    col for col in X_fit.select_dtypes(include=[np.number]).columns
    if col not in account_cols and col != 'Is Laundering'
]

categorical_features = ['Payment Format', 'Receiving Currency', 'Payment Currency']

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), categorical_features),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

X_fit = preprocessor.fit_transform(X_fit)
X_test = preprocessor.transform(X_test)
X_val = preprocessor.transform(X_val)

In [10]:
log = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=400, tol=1e-5)
log.fit(X_fit, y_fit, sample_weight=sample_weight)

scores = log.predict_proba(X_test)[:, 1]
precision, recall, _ = precision_recall_curve(y_test, scores)
roc_auc_score(y_test, scores), average_precision_score(y_test, scores)

(0.9585963592709781, 0.33151147520379376)

(0.8376901282090947, 0.003561472074727589)

(0.861696146976858, 0.018676374512108074)

(0.9427477107093325, 0.17615445221361023)

(0.9636140905753943, 0.260369221138288)

0.3320342015753189

0.3302096102029402 - 0.0212

(0.9585325158532447, 0.32895158688429216) - 0.1

(0.9585963592709781, 0.33151147520379376) - 0.3900

In [43]:
import optuna
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score

# Объединяем исходные train и val (85% данных) для кросс-валидации
# Временной ряд сохраняет естественную сортировку из датасета
cv_mask = train_mask | val_mask
X_cv = df.loc[cv_mask].drop(columns='Is Laundering').reset_index(drop=True)
y_cv = df.loc[cv_mask, 'Is Laundering'].reset_index(drop=True)

def objective(trial):
    neg_ratio = trial.suggest_float("neg_subsample_ratio", 0.01, 0.6, step=0.01)
    
    # Разбиваем временной ряд на 3 последовательных фолда
    # F1: train [0-25%], val [25-50%]
    # F2: train [0-50%], val [50-75%]
    # F3: train [0-75%], val [75-100%]
    tscv = TimeSeriesSplit(n_splits=3)
    fold_scores = []
    
    for train_idx, val_idx in tscv.split(X_cv):
        X_fold_train, y_fold_train = X_cv.iloc[train_idx], y_cv.iloc[train_idx]
        X_fold_val, y_fold_val = X_cv.iloc[val_idx], y_cv.iloc[val_idx]
        
        # 1. Изолированное сэмплирование строго внутри обучающей части текущего фолда
        pos_mask = y_fold_train == 1
        neg_mask = y_fold_train == 0
        
        pos_index = y_fold_train.index[pos_mask]
        neg_index = y_fold_train.index[neg_mask]
        n_total_neg = len(neg_index)
        
        rng = np.random.default_rng(42)
        n_sample_neg = max(1, int(neg_ratio * n_total_neg))
        sampled_neg_idx = rng.choice(neg_index, size=n_sample_neg, replace=False)
        
        neg_weight = float(n_total_neg) / n_sample_neg
        fit_idx = np.concatenate([pos_index, sampled_neg_idx])
        
        X_fit = X_fold_train.loc[fit_idx]
        y_fit = y_fold_train.loc[fit_idx]
        sample_weight = np.where(y_fit.to_numpy() == 0, neg_weight, 1.0).astype(np.float64)
        
        # 2. Изолированный препроцессинг для каждого фолда во избежание Data Leakage
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), numeric_features),
                ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), categorical_features),
            ],
            remainder="drop",
            sparse_threshold=1.0,
        )
        
        X_fit_prep = preprocessor.fit_transform(X_fit)
        X_val_prep = preprocessor.transform(X_fold_val)
        
        # 3. Обучение и оценка
        log = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=400, tol=1e-5)
        log.fit(X_fit_prep, y_fit, sample_weight=sample_weight)
        
        scores = log.predict_proba(X_val_prep)[:, 1]
        fold_scores.append(average_precision_score(y_fold_val, scores))
        
    # Оптимизируем среднее значение PR-AUC по всем временным окнам
    return np.mean(fold_scores)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"Best neg_subsample_ratio: {study.best_params['neg_subsample_ratio']:.4f}")
print(f"Best Rolling CV PR-AUC:   {study.best_value:.4f}")

  0%|          | 0/30 [00:00<?, ?it/s]

Best neg_subsample_ratio: 0.3900
Best Rolling CV PR-AUC:   0.1169
